# Kernels and Structures

This page summarizes the mathematical background of the kernels in `smudgy`: fixed-scale assignment kernels, whose size is set by the grid, adaptive SPH kernels, whose size is set by the local particle density, and their anisotropic (covariant) generalization, which also adapts the kernel shape.

Throughout, particles carry an index $i$, positions $\mathbfit{x}_i$, masses $m_i$ (the `weights` passed to {py:class}`~smudgy.pointcloud.PointCloud`), densities $\rho_i$ and field values $f_i$. A query position is denoted by $\mathbfit{x}$, its neighbourhood of $N_{\rm ngb}$ (`num_neighbors`) nearest particles by $\mathcal{N}$, and $\Delta\mathbfit{x}_i = \mathbfit{x} - \mathbfit{x}_i$.

## Fixed-kernel assignment

The simplest way to turn particles into a field is to spread each particle over a fixed neighbourhood of cells of a uniform Cartesian grid. For a grid of $N_{\rm c}$ cells per axis covering a $d$-dimensional box of side $L$, the cell spacing is $s = L/N_{\rm c}$, and the density deposited in cell $g$ is

$$
\rho(\mathbfit{x}_{g}) = \frac{1}{s^{d}} \sum_{i} m_{i} \prod_{j=1}^{d} W_{p}\!\left( \frac{x_{g,j}-x_{i,j}}{s} \right),
$$

where $W_p$ is a one-dimensional assignment function. `smudgy` implements the cardinal B-splines of order $p$, i.e. the unit top-hat convolved with itself $p-1$ times. Order $p$ spans $p$ cells per axis and is $C^{p-2}$ continuous. The weights are non-negative and form a partition of unity, so mass is conserved exactly.

```{list-table}
:header-rows: 1

* - Scheme
  - `kernel_name`
  - $p$
  - Cells/axis
  - Support
  - Continuity
* - Nearest grid point (NGP)
  - `'ngp'`
  - 1
  - 1
  - $\pm 0.5\,s$
  - discontinuous
* - Cloud-in-cell (CIC)
  - `'cic'`
  - 2
  - 2
  - $\pm 1.0\,s$
  - $C^{0}$
* - Triangular-shaped cloud (TSC)
  - `'tsc'`
  - 3
  - 3
  - $\pm 1.5\,s$
  - $C^{1}$
* - Cubic B-spline
  - `'bspline4'`
  - 4
  - 4
  - $\pm 2.0\,s$
  - $C^{2}$
* - Quartic B-spline
  - `'bspline5'`
  - 5
  - 5
  - $\pm 2.5\,s$
  - $C^{3}$
* - Quintic B-spline
  - `'bspline6'`
  - 6
  - 6
  - $\pm 3.0\,s$
  - $C^{4}$
```

Since the deposition is a convolution followed by sampling, the smoothing it introduces is known analytically: the Fourier transform of the assignment function is

$$
\widetilde{W}_{p}(\mathbfit{k}) = \prod_{j=1}^{d} \left[ \frac{\sin\!\left(k_{j}s/2\right)}{k_{j}s/2} \right]^{p},
$$

so every additional order adds one more sinc factor. These schemes are selected with `adaptive=False` in {py:meth}`~smudgy.pointcloud.PointCloud.deposit`. Their common limitation is that the kernel size is set by the grid rather than by the particles: dense regions are oversmoothed, while underdense regions remain dominated by discreteness noise. See the {doc}`fixed vs. adaptive tutorial <../tutorials/deposition/fixed_adaptive_methods>` for a visual comparison.

## Adaptive SPH kernels

Smoothed Particle Hydrodynamics (SPH) starts from the identity $f(\mathbfit{x}) = \int f(\mathbfit{x}') \, \delta(\mathbfit{x} - \mathbfit{x}') \, \mathrm d \mathbfit{x}'$ and replaces the Dirac delta by a smoothing kernel $W$,

$$
f(\mathbfit{x}) \approx \int_{\Omega} f(\mathbfit{x}') \, W(\mathbfit{x} - \mathbfit{x}', h) \, \mathrm d \mathbfit{x}'.
$$

The smoothing length $h$ sets the radius $n \times h$ of the support $\Omega$, with a kernel-dependent support multiple $n$. The kernel is positive, symmetric, normalized to unity and, in the classical formulation, isotropic,

$$
W(\mathbfit{x}-\mathbfit{x}', h) = \frac{\alpha_d}{h^d} K(q)
\quad \text{with}\quad
q = \frac{\lVert\mathbfit{x}-\mathbfit{x}'\rVert}{h},
$$

where $K$ is a dimensionless radial kernel function and $\alpha_d$ a dimension-dependent normalization constant. In `smudgy`, each particle carries its own smoothing length $h_i$, set by the distance to its $N_{\rm ngb}$-th nearest neighbour, so that the resolution follows the local density.

`smudgy` offers three **structures** that define how the kernel adapts to the particle distribution:

+ `separable`: tensor products of one-dimensional kernels with per-particle scale $h_i$ along each axis (axis-aligned boxes), $W(\Delta\mathbfit{x}, h) = \prod_{a=1}^{d} W_{\rm 1D}(\Delta x_a, h)$. Their cell integrals factorize into one-dimensional integrals, which are evaluated exactly. Available for deposition only, with the `tophat`, `tsc` and `gaussian` kernels.
+ `isotropic`: radial kernels with spherical support, as defined above.
+ `covariant`: the same radial kernels with ellipsoidal support, defined by a smoothing tensor (see below).

The interpolation and density estimates based on these kernels are described in the {doc}`overview <overview>`.

## Anisotropic (covariant) kernels

An isotropic kernel adapts to the local particle density only through its size. In strongly anisotropic regions, such as filaments, sheets or shocks, it oversmooths along the compressed direction. The `covariant` structure therefore promotes the smoothing length to a smoothing tensor $\mathbfss{H}$ with ellipsoidal support, following the purely covariance-based approach of Marinho (2021), which uses only the spatial information of the particle distribution.

### Local covariance and Mahalanobis distance

For the neighbourhood $\mathcal{N}$ of a position $\mathbfit{x}$ (a particle or an arbitrary query point) with total mass $M = \sum_{i \in \mathcal{N}} m_i$, the local covariance tensor

$$
\bmath{\Sigma}(\mathbfit{x}) = \frac{1}{M} \sum_{i \in \mathcal{N}} m_i \, \Delta \mathbfit{x}_i \Delta \mathbfit{x}_i^\top
$$

describes how the neighbourhood mass is spread and oriented about $\mathbfit{x}$. Its eigendecomposition,

$$
\bmath{\Sigma} = \mathbfss{Q} \bmath{\Lambda} \mathbfss{Q}^\top, \qquad \bmath{\Lambda} = \operatorname{diag}(\sigma_1^2,\ldots,\sigma_d^2),
$$

yields the principal variances $\sigma_a^2$ and the principal axes (the columns of $\mathbfss{Q}$). $\bmath{\Sigma}$ induces the Mahalanobis distance

$$
D^2(\mathbfit{x},\mathbfit{x}_i) = \Delta\mathbfit{x}_i^\top \bmath{\Sigma}^{-1} \Delta\mathbfit{x}_i,
$$

which measures separations relative to the local spread in each direction.

### The smoothing tensor

$\bmath{\Sigma}^{1/2}$ fixes the shape and orientation of the kernel but not its size. The size is set by requiring the kernel to enclose all neighbours,

$$
\mathbfss{H} = \mathbfss{Q} \operatorname{diag}(h_1,\ldots,h_d) \, \mathbfss{Q}^\top = \zeta_{\max}\, \bmath{\Sigma}^{1/2},
\qquad
\zeta_{\max} = \max_{i\in\mathcal{N}} D(\mathbfit{x},\mathbfit{x}_i),
$$

with principal smoothing lengths $h_a = \zeta_{\max}\sigma_a$. The ellipsoid $\lVert\mathbfss{H}^{-1}\Delta\mathbfit{x}\rVert \le 1$ is thus the smallest ellipsoid of the local shape that contains all $N_{\rm ngb}$ neighbours; for an isotropic neighbourhood, every axis reduces to the ordinary smoothing length $h$. In the normalized coordinate $\bmath{\xi}_i = \mathbfss{H}^{-1}\Delta\mathbfit{x}_i$, the anisotropic kernel and its gradient read

$$
W(\Delta\mathbfit{x}_i,\mathbfss{H}_i) = \frac{\alpha_d}{\det\mathbfss{H}_i} K(\xi_i),
\qquad
\nabla W(\Delta\mathbfit{x}_i,\mathbfss{H}_i) = \frac{\alpha_d}{\det\mathbfss{H}_i} K'(\xi_i) \, \frac{\mathbfss{H}_i^{-2}\Delta\mathbfit{x}_i}{\xi_i},
$$

i.e. all isotropic relations carry over by replacing $q_i \rightarrow \xi_i$ and $h^d \rightarrow \det\mathbfss{H}$.

### Volume-preserving smoothing tensor

Under the hull condition above, the kernel volume $\det\mathbfss{H} = \zeta_{\max}^{d}\,(\det\bmath{\Sigma})^{1/2}$ is set by the neighbourhood itself, so the covariant kernel generally differs from the isotropic one in both shape and size. The volume-preserving formulation rescales the axes to the volume of the isotropic kernel of the same particle,

$$
\mathbfss{H}_{\rm vp} = \frac{h}{(\det\mathbfss{H})^{1/d}}\,\mathbfss{H}
\quad\Rightarrow\quad
\det\mathbfss{H}_{\rm vp} = h^{d},
$$

where $h$ is the isotropic smoothing length. It differs from the isotropic kernel only in shape and orientation, but may no longer enclose all neighbours.

### Options in `smudgy`

Three keyword arguments of {py:meth}`~smudgy.pointcloud.PointCloud.compute_smoothing` control $\mathbfss{H}$:

+ `ratio_max` (default `100`) caps the axis ratio $h_{\max}/h_{\min}$, which guards against (near-)coplanar or collinear neighbour sets.
+ `min_axis` sets a lower bound on every axis $h_a$, e.g. a softening length or the cell size.
+ `volume_preserving=True` switches to $\mathbfss{H}_{\rm vp}$.

:::{note}
The neighbourhood $\mathcal{N}$ is found with a standard (Euclidean) nearest-neighbour search and is not iteratively refined in the Mahalanobis metric.
:::

## Kernel support in interpolation and deposition

Interpolation and density estimates only sum over the $N_{\rm ngb}$ nearest neighbours. Kernels whose support multiple exceeds $n = 1$ (e.g. `gaussian`, `quintic_spline`, `tsc` and the Wendland kernels) are therefore rescaled so that their support ends at the smoothing length or on the smoothing-tensor ellipsoid ($h \rightarrow h/n$, $\mathbfss{H} \rightarrow \mathbfss{H}/n$). Deposition instead uses the full footprint of radius $n \times h$, so these kernels have different physical widths in the two modes.

## Available kernels

Each kernel $W = \alpha_d\, K(q)/h^d$ (or $\alpha_d\, K(\xi)/\det\mathbfss{H}$ in the covariant case) is defined by its dimensionless profile $K(q)$, its support multiple $n$, beyond which $K$ vanishes, and its normalization constant $\alpha_d$. Besides the profile itself, every kernel provides its analytic derivative (for the gradient, divergence and curl modes of the interpolation) and its radial antiderivative (for the exact shell integrals of the deposition sample tables, see {doc}`numerical quadrature <numerical_quadrature>`).

```{list-table}
:header-rows: 1
:class: wide-table

* - Kernel
  - `kernel_name`
  - $K(q)$
  - Support $n$
  - $\alpha_\mathrm{1D}$
  - $\alpha_\mathrm{2D}$
  - $\alpha_\mathrm{3D}$
  - Separable

* - Top-hat
  - `'tophat'`
  - $\begin{cases}1 & q \le 0.5 \\ 0 & q > 0.5\end{cases}$
  - $0.5$
  - $1$
  - $4/\pi$
  - $6/\pi$
  - ✓

* - Triangular
  - `'tsc'`
  - $\begin{cases}1-\frac{q}{1.5} & q < 1.5 \\ 0 & q \ge 1.5\end{cases}$
  - $1.5$
  - $2/3$
  - $4/(3\pi)$
  - $8/(9\pi)$
  - ✓

* - Gaussian
  - `'gaussian'`
  - $e^{-q^2}$
  - $3.0$
  - $1/\sqrt{\pi}$
  - $1/\pi$
  - $1/\pi^{3/2}$
  - ✓

* - Lucy
  - `'lucy'`
  - $\begin{cases}(1+3q)(1-q)^3 & q \le 1 \\ 0 & q > 1\end{cases}$
  - $1.0$
  - $5/4$
  - $5/\pi$
  - $105/(16\pi)$
  - ✗

* - Cubic spline
  - `'cubic_spline'`
  - $\begin{cases}1 - 6q^2 + 6q^3 & 0 \le q < 0.5 \\ 2(1-q)^3 & 0.5 \le q < 1 \\ 0 & q \ge 1\end{cases}$
  - $1.0$
  - $4/3$
  - $40/(7\pi)$
  - $8/\pi$
  - ✗

* - Quintic spline
  - `'quintic_spline'`
  - $\begin{cases}(3-q)^5 - 6(2-q)^5 + 15(1-q)^5 & 0 \le q < 1 \\ (3-q)^5 - 6(2-q)^5 & 1 \le q < 2 \\ (3-q)^5 & 2 \le q < 3 \\ 0 & q \ge 3\end{cases}$
  - $3.0$
  - $1/120$
  - $7/(478\pi)$
  - $1/(120\pi)$
  - ✗

* - Wendland $C^2$
  - `'wendland_c2'`
  - $\begin{cases}(1-q/2)^4(2q+1) & q \le 2 \\ 0 & q > 2\end{cases}$
  - $2.0$
  - $5/8$
  - $7/(4\pi)$
  - $21/(16\pi)$
  - ✗

* - Wendland $C^4$
  - `'wendland_c4'`
  - $\begin{cases}(1-q/2)^6\left(\frac{35}{12}q^2+3q+1\right) & q \le 2 \\ 0 & q > 2\end{cases}$
  - $2.0$
  - $3/4$
  - $9/(4\pi)$
  - $495/(256\pi)$
  - ✗

* - Wendland $C^6$
  - `'wendland_c6'`
  - $\begin{cases}(1-q/2)^8\left(4q^3+\frac{25}{4}q^2+4q+1\right) & q \le 2 \\ 0 & q > 2\end{cases}$
  - $2.0$
  - $55/64$
  - $39/(14\pi)$
  - $1365/(512\pi)$
  - ✗
```

$q$ denotes the normalized distance $\lVert\Delta\mathbfit{x}\rVert/h$, or $\xi$ for covariant kernels. The Gaussian is truncated at $q=3$. For the Wendland kernels, $K(q)$ is given in its two- and three-dimensional form; in one dimension, the corresponding one-dimensional Wendland functions are used, to which the listed $\alpha_{\mathrm{1D}}$ refer. Kernels marked as separable are additionally available as tensor products of their one-dimensional profiles for the `separable` structure.